# 05-01 · SWMM Ensemble Runner — Climate Change × Spatial Uncertainty

Runs SWMM for all combinations of:
- **12 Pareto members** (calibrated parameter sets)
- **41 WRF events** × **2 conditions** (historical / future)
- **81 spatial shifts** (y = −20 km to +20 km, 500 m step; x = 0)

**Total: 79,704 simulations.** Results for each member are saved to a separate pickle
in `outputs/climate/raw/`.  If a pickle already exists the member is skipped, so the
cell can be re-run safely after an interruption.


In [9]:
import os
import re
import sys
import time
import logging
from datetime import datetime, timedelta
from pathlib import Path

import numpy as np
import pandas as pd
from swmm_api.input_file import read_inp_file

sys.path.insert(0, str(Path(r"D:\MY_CODES\UrbanRunoffModeling_Refactored")))
from urban_runoff.swmm.inp_editor import apply_all_factors
from urban_runoff.swmm.runner     import run_simulation
from urban_runoff.data.loaders    import load_swmm_output

logging.basicConfig(level=logging.WARNING,
                    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
                    datefmt="%H:%M:%S")

# ── Read-only inputs ──────────────────────────────────────────────────────────
# Original uncalibrated SWMM model (storage_imperv = 1.8 mm, pre-calibration).
# Must NOT be Final.inp (which is post-calibration): apply_all_factors would
# double-multiply Type-B parameters that Final.inp already embeds.
BASE_INP_PATH = Path(r"D:\Development\RESEARCH\Raanana\SWMM\from_radar\Cross_validation\2012_01_13\raanana_28subcatchments.inp")
WRF_TXT_ROOT  = Path(r"\\vscifs\hydrolab1\hydrolab\home\Raz\WRF\txtfiles")
PARETO_CSV    = Path(r"D:\MY_CODES\UrbanRunoffModeling_Refactored\outputs\pareto\final\pareto_ensemble_full.csv")

# ── Output paths ──────────────────────────────────────────────────────────────
OUTPUT_DIR    = Path(r"D:\MY_CODES\UrbanRunoffModeling_Refactored\outputs\climate\raw")
TEMP_DIR      = Path(r"D:\MY_CODES\UrbanRunoffModeling_Refactored\outputs\climate\temp")
WORK_INP      = TEMP_DIR / "pareto_runner.inp"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TEMP_DIR.mkdir(parents=True, exist_ok=True)

# ── Simulation dimensions ─────────────────────────────────────────────────────
EVENTS_L       = [f"{i:02}" for i in range(1, 42)]          # '01' through '41'
CONDITIONS     = ["historical", "future"]
X_SHIFT        = 0
Y_SHIFTS       = np.arange(-20000, 20500, 500).astype(int)  # 81 values
BASIN_AREA_M2  = 1325.6 * 1e4                               # 13,256,000 m²

# ── Pareto ensemble ───────────────────────────────────────────────────────────
PARETO_INDICES = [682, 701, 708, 709, 710, 717, 718, 719, 727, 728, 898, 917]
FACTOR_COLS    = ["imperviousness", "storage", "width", "n",
                  "pct_zero", "cn", "pct_routed", "evaporation"]

# Maps CSV column headers (from pareto_ensemble_full.csv) to the canonical
# parameter names expected by apply_all_factors.
CSV_TO_FACTOR = {
    "IMP":        "imperviousness",
    "Storage":    "storage",
    "Width":      "width",
    "N":          "n",
    "PCT_ZERO":   "pct_zero",
    "CN":         "cn",
    "PCT_ROUTED": "pct_routed",
    "EVAP":       "evaporation",
}

# ── Urbanization scenario control ─────────────────────────────────────────────
# Add new multipliers here to extend the analysis; already-computed
# multipliers in an existing pickle are skipped automatically.
URBAN_MULTIPLIERS = [1.0, 2.0]

# Pilot run: only the optimum member to verify CDF alignment against the legacy
# pipeline before committing to the full ~30-hour ensemble run.
# Change to PARETO_INDICES to run all 12 members once the pilot is verified.
TARGET_MEMBERS = PARETO_INDICES

# ── Pickle column schema helpers ──────────────────────────────────────────────
_ID_COLS = [
    ("event_num", "", "", ""),
    ("x",         "", "", ""),
    ("y",         "", "", ""),
    ("d",         "", "", ""),
]

def _urban_data_cols(urban_mult):
    """Return the 8 data column 4-tuples for one urbanization multiplier."""
    pairs = [
        ("basin discharge", "max_discharge"),
        ("basin discharge", "total_discharge"),
        ("basin discharge", "runoff_coefficient"),
        ("full_basin",      "total_rain"),
    ]
    return [(urban_mult, cond, cat, metric)
            for cond in ("historical", "future")
            for cat, metric in pairs]


In [10]:
# ── DATA PURGE — run once to delete all corrupted pickles ────────────────────
# All existing pickles were generated using Final.inp (post-calibration) as
# the base model. This caused storage_imperv to be double-multiplied by the
# Pareto storage factor (e.g., 12.6 × 7.0 = 88.2 mm instead of 12.6 mm for
# member 708). All pickles must be deleted and regenerated from the correct
# uncalibrated base.
# Run this cell once before running the notebook; safe to skip if raw/ is empty.
_pkl_files = sorted(OUTPUT_DIR.glob("pareto_*_results.pkl"))
if not _pkl_files:
    print("No pickles found — nothing to delete.")
else:
    print(f"Deleting {len(_pkl_files)} corrupted pickle(s):")
    for _p in _pkl_files:
        _p.unlink()
        print(f"  Deleted: {_p.name}")
    print("Purge complete.")


Deleting 1 corrupted pickle(s):
  Deleted: pareto_708_results.pkl
Purge complete.


In [11]:
# ── Section 0: Schema migration (run once, idempotent) ────────────────────────
# Upgrades existing pickles from a 3-level to a 4-level column MultiIndex.
# Identifier columns ("event_num", "x", "y", "d") gain an empty 4th level.
# Data columns gain urban_multiplier = 1.0 as level 0.

def _upgrade_pickle_schema(pkl_path):
    df = pd.read_pickle(str(pkl_path))
    if df.columns.nlevels == 4:
        print(f"  {pkl_path.name}: already 4-level, skipping")
        return
    _ID_KEYS = {"event_num", "x", "y", "d"}
    def _up(col):
        a, b, c = col
        if a in _ID_KEYS:
            return (a, b, c, "")
        return (1.0, a, b, c)
    df.columns = pd.MultiIndex.from_tuples([_up(c) for c in df.columns])
    df.to_pickle(str(pkl_path))
    print(f"  {pkl_path.name}: upgraded from 3-level to 4-level")

print("Migrating existing pickles ...")
for _idx in PARETO_INDICES:
    _p = OUTPUT_DIR / f"pareto_{_idx}_results.pkl"
    if _p.exists():
        _upgrade_pickle_schema(_p)
    else:
        print(f"  pareto_{_idx}_results.pkl: not found, skipping migration")
print("Migration complete.")


Migrating existing pickles ...
  pareto_682_results.pkl: not found, skipping migration
  pareto_701_results.pkl: not found, skipping migration
  pareto_708_results.pkl: not found, skipping migration
  pareto_709_results.pkl: not found, skipping migration
  pareto_710_results.pkl: not found, skipping migration
  pareto_717_results.pkl: not found, skipping migration
  pareto_718_results.pkl: not found, skipping migration
  pareto_719_results.pkl: not found, skipping migration
  pareto_727_results.pkl: not found, skipping migration
  pareto_728_results.pkl: not found, skipping migration
  pareto_898_results.pkl: not found, skipping migration
  pareto_917_results.pkl: not found, skipping migration
Migration complete.


In [12]:
# ── Verify paths before starting ──────────────────────────────────────────────
assert BASE_INP_PATH.exists(), f"Base inp not found: {BASE_INP_PATH}"
assert WRF_TXT_ROOT.exists(),  f"WRF txt root not found: {WRF_TXT_ROOT}"
assert PARETO_CSV.exists(),    f"Pareto CSV not found: {PARETO_CSV}"

n_per_member = len(EVENTS_L) * len(CONDITIONS) * len(Y_SHIFTS)
print(f"Simulations per member : {n_per_member:,}")
print(f"Target members         : {TARGET_MEMBERS}")
print(f"Total simulations      : {len(TARGET_MEMBERS) * n_per_member:,}  (pilot)")


Simulations per member : 6,642
Target members         : [682, 701, 708, 709, 710, 717, 718, 719, 727, 728, 898, 917]
Total simulations      : 79,704  (pilot)


In [13]:
def _read_rain_shift_txt(directory, x_shift, y_shift):
    """Return the content of the WRF rainfall txt file for the given spatial shift."""
    for fname in os.listdir(directory):
        if "rain_shift_" not in fname:
            continue
        xm = re.search(r"x(plus|minus)_(\d+)", fname)
        ym = re.search(r"y(plus|minus)_(\d+)", fname)
        if not xm or not ym:
            continue
        fx = int(xm.group(2)) * (1 if xm.group(1) == "plus" else -1)
        fy = int(ym.group(2)) * (1 if ym.group(1) == "plus" else -1)
        if fx == x_shift and fy == y_shift:
            with open(os.path.join(directory, fname), "r") as fh:
                return fh.read()
    raise FileNotFoundError(
        f"No WRF txt found for x={x_shift}, y={y_shift} in {directory}"
    )


def _update_timeseries_and_options(inp, timeseries_text):
    """Assign WRF timeseries text to SWMM inp and update simulation OPTIONS dates."""
    lines     = timeseries_text.strip().split("\n")
    data_rows = [l for l in lines if not l.startswith(";;")]

    def _parse_row(row):
        cols = row.split()
        d = datetime.strptime(cols[1], "%m/%d/%Y").date()
        t = datetime.strptime(cols[2], "%H:%M:%S").time()
        return d, t

    first_d, first_t   = _parse_row(data_rows[0])
    second_d, second_t = _parse_row(data_rows[1])
    last_d,  last_t    = _parse_row(data_rows[-1])

    t1 = timedelta(hours=first_t.hour,  minutes=first_t.minute,  seconds=first_t.second)
    t2 = timedelta(hours=second_t.hour, minutes=second_t.minute, seconds=second_t.second)
    interval_str = str(t2 - t1)

    start_dt = datetime.combine(first_d, first_t)
    if start_dt.hour < 6:
        start_dt -= timedelta(days=1)
    end_dt = datetime.combine(last_d, last_t)
    if end_dt.hour >= 18:
        end_dt += timedelta(days=1)

    inp["OPTIONS"]["START_DATE"]        = start_dt.date()
    inp["OPTIONS"]["START_TIME"]        = (start_dt - timedelta(hours=6)).time()
    inp["OPTIONS"]["REPORT_START_DATE"] = start_dt.date()
    inp["OPTIONS"]["REPORT_START_TIME"] = (start_dt - timedelta(hours=6)).time()
    inp["OPTIONS"]["END_DATE"]          = end_dt.date()
    inp["OPTIONS"]["END_TIME"]          = (end_dt + timedelta(hours=6)).time()

    inp["TIMESERIES"] = timeseries_text

    n_ts = len(inp["TIMESERIES"].keys())
    n_rg = len(inp["RAINGAGES"].keys())
    if n_ts != n_rg:
        raise ValueError(f"TIMESERIES count ({n_ts}) != RAINGAGES count ({n_rg})")

    ts_keys = list(inp["TIMESERIES"].keys())
    for i, rg_key in enumerate(inp["RAINGAGES"].keys()):
        ts_name = inp["TIMESERIES"][ts_keys[i]]["name"]
        inp["RAINGAGES"][rg_key]["timeseries"] = ts_name
        inp["RAINGAGES"][rg_key]["interval"]   = interval_str
        inp["RAINGAGES"][rg_key]["form"]        = "INTENSITY"


def _infer_timestep_s(sim_df):
    """Return the simulation timestep in seconds from the output DataFrame index."""
    if len(sim_df) < 2:
        return 300.0
    return float((sim_df.index[1] - sim_df.index[0]).total_seconds())


def _sim_stats(sim_df, dt_s):
    """Return (max_discharge, total_discharge_m3, total_rain_mm, runoff_coeff)."""
    max_disc    = float(sim_df["SWMM outflow [CMS]"].max())
    total_disc  = float(sim_df["SWMM outflow [CMS]"].sum()) * dt_s
    total_rain  = float(sim_df["rainfall [mm/h]"].sum()) * (dt_s / 3600.0)
    rain_vol_m3 = max(total_rain / 1000.0 * BASIN_AREA_M2, 1e-9)
    return max_disc, total_disc, total_rain, total_disc / rain_vol_m3


def _apply_urban_multiplier(inp, urban_mult):
    """Scale each subcatchment's imperviousness by urban_mult, capped at 100%.
    Must be called AFTER apply_all_factors so the Pareto factor is already applied."""
    for sc in inp["SUBCATCHMENTS"].values():
        sc.imperviousness = min(sc.imperviousness * urban_mult, 100.0)


print("Helper functions defined.")


Helper functions defined.


In [14]:
pareto_df  = pd.read_csv(str(PARETO_CSV), header=[0, 1], index_col=0)
factors_df = pareto_df["Factors"].rename(columns=CSV_TO_FACTOR)

print(f"Pareto members loaded : {len(factors_df)}")
print(f"Factor columns        : {list(factors_df.columns)}")
print(f"\nUrbanization multipliers to run: {URBAN_MULTIPLIERS}")
print(f"\nPer-member status:")
for _idx in PARETO_INDICES:
    _p = OUTPUT_DIR / f"pareto_{_idx}_results.pkl"
    if not _p.exists():
        print(f"  pareto_{_idx}: no pickle (baseline not yet run)")
        continue
    _df = pd.read_pickle(str(_p))
    _done_mults = sorted(m for m in _df.columns.get_level_values(0) if isinstance(m, float))
    _todo_mults = [m for m in URBAN_MULTIPLIERS if m not in _done_mults]
    print(f"  pareto_{_idx}: done={_done_mults}  todo={_todo_mults}")


Pareto members loaded : 12
Factor columns        : ['imperviousness', 'storage', 'width', 'n', 'pct_zero', 'cn', 'pct_routed', 'evaporation']

Urbanization multipliers to run: [1.0, 2.0]

Per-member status:
  pareto_682: no pickle (baseline not yet run)
  pareto_701: no pickle (baseline not yet run)
  pareto_708: no pickle (baseline not yet run)
  pareto_709: no pickle (baseline not yet run)
  pareto_710: no pickle (baseline not yet run)
  pareto_717: no pickle (baseline not yet run)
  pareto_718: no pickle (baseline not yet run)
  pareto_719: no pickle (baseline not yet run)
  pareto_727: no pickle (baseline not yet run)
  pareto_728: no pickle (baseline not yet run)
  pareto_898: no pickle (baseline not yet run)
  pareto_917: no pickle (baseline not yet run)


In [15]:
for pareto_idx in TARGET_MEMBERS:
    pkl_path    = OUTPUT_DIR / f"pareto_{pareto_idx}_results.pkl"
    factor_dict = factors_df.loc[pareto_idx, FACTOR_COLS].to_dict()

    if pkl_path.exists():
        df_existing = pd.read_pickle(str(pkl_path))
        done_mults  = {m for m in df_existing.columns.get_level_values(0) if isinstance(m, float)}
    else:
        df_existing = None
        done_mults  = set()

    todo_mults = [m for m in URBAN_MULTIPLIERS if m not in done_mults]

    if not todo_mults:
        print(f"pareto_{pareto_idx}: all multipliers {URBAN_MULTIPLIERS} already done, skipping.")
        continue

    for urban_mult in todo_mults:
        print(f"\n{'='*60}")
        print(f"Member {pareto_idx} | urban_mult={urban_mult:.1f} | {factor_dict}")
        t0 = time.time()

        # Load fresh inp, apply Pareto calibration, then apply urbanization.
        # Order matters: apply_all_factors sets parameters from the uncalibrated
        # base; _apply_urban_multiplier then scales imperviousness and caps at 100%.
        inp = read_inp_file(str(BASE_INP_PATH))
        apply_all_factors(inp, factor_dict)
        _apply_urban_multiplier(inp, urban_mult)
        # Base file (raanana_28subcatchments.inp) has REPORT_STEP=5 min.
        # Override to 10 min to match the legacy Final.inp output resolution so
        # that _infer_timestep_s returns 600 s and max_discharge sampling is identical.
        inp["OPTIONS"]["REPORT_STEP"] = timedelta(minutes=10)

        new_rows = []
        id_rows  = []   # identifier rows; populated only when building a fresh pickle
        n_done   = 0

        for event_num in EVENTS_L:
            hist_stats: dict = {}
            fut_stats:  dict = {}

            for condition, store in [("historical", hist_stats), ("future", fut_stats)]:
                wrf_dir = str(WRF_TXT_ROOT / condition / f"event_{event_num}")

                for y_shift in Y_SHIFTS:
                    rain_txt = _read_rain_shift_txt(wrf_dir, X_SHIFT, y_shift)
                    _update_timeseries_and_options(inp, rain_txt)
                    inp.write_file(str(WORK_INP))
                    run_simulation(WORK_INP)
                    sim_df = load_swmm_output(WORK_INP.with_suffix(".out"))
                    dt_s   = _infer_timestep_s(sim_df)
                    store[y_shift] = _sim_stats(sim_df, dt_s)
                    n_done += 1
                    if n_done % 500 == 0:
                        elapsed  = time.time() - t0
                        rate     = n_done / elapsed
                        eta_min  = (n_per_member - n_done) / rate / 60
                        print(f"  [{n_done:>5}/{n_per_member}] ev={event_num} "
                              f"cond={condition} y={y_shift:+7d} | "
                              f"{rate:.1f} runs/s | ETA {eta_min:.0f} min")

            for y_shift in Y_SHIFTS:
                h = hist_stats[y_shift]
                f = fut_stats[y_shift]
                new_rows.append([h[0], h[1], h[3], h[2],
                                 f[0], f[1], f[3], f[2]])
                if df_existing is None:
                    id_rows.append([int(event_num), X_SHIFT, y_shift, 0])

        new_df = pd.DataFrame(
            new_rows,
            columns=pd.MultiIndex.from_tuples(_urban_data_cols(urban_mult)),
        )

        if df_existing is None:
            # First multiplier for a fresh pickle: prepend identifier columns.
            id_df       = pd.DataFrame(id_rows,
                              columns=pd.MultiIndex.from_tuples(_ID_COLS))
            df_existing = pd.concat([id_df, new_df], axis=1)
        else:
            df_existing = pd.concat([df_existing, new_df], axis=1)

        df_existing.to_pickle(str(pkl_path))
        elapsed_min = (time.time() - t0) / 60
        print(f"  Saved: {pkl_path.name}  ({elapsed_min:.1f} min, "
              f"now {df_existing.shape[1]} cols)")

print("\nAll runs complete.")


Member 682 | urban_mult=1.0 | {'imperviousness': 1.0, 'storage': 5.0, 'width': 0.7, 'n': 1.0, 'pct_zero': 1.0, 'cn': 1.0, 'pct_routed': 40.0, 'evaporation': 4.0}
  [  500/6642] ev=04 cond=historical y= -13500 | 1.0 runs/s | ETA 100 min
  [ 1000/6642] ev=07 cond=historical y=  -6500 | 0.5 runs/s | ETA 189 min
  [ 1500/6642] ev=10 cond=historical y=   +500 | 0.5 runs/s | ETA 156 min
  [ 2000/6642] ev=13 cond=historical y=  +7500 | 0.6 runs/s | ETA 135 min
  [ 2500/6642] ev=16 cond=historical y= +14500 | 0.6 runs/s | ETA 112 min
  [ 3000/6642] ev=19 cond=future y= -19000 | 0.6 runs/s | ETA 94 min
  [ 3500/6642] ev=22 cond=future y= -12000 | 0.7 runs/s | ETA 76 min
  [ 4000/6642] ev=25 cond=future y=  -5000 | 0.7 runs/s | ETA 61 min
  [ 4500/6642] ev=28 cond=future y=  +2000 | 0.8 runs/s | ETA 47 min
  [ 5000/6642] ev=31 cond=future y=  +9000 | 0.8 runs/s | ETA 34 min
  [ 5500/6642] ev=34 cond=future y= +16000 | 0.8 runs/s | ETA 23 min
  [ 6000/6642] ev=38 cond=historical y= -17500 | 0.8 

In [16]:
print("Summary of completed pickles:")
for pareto_idx in PARETO_INDICES:
    pkl_path = OUTPUT_DIR / f"pareto_{pareto_idx}_results.pkl"
    if pkl_path.exists():
        df_tmp = pd.read_pickle(str(pkl_path))
        print(f"  pareto_{pareto_idx}: {df_tmp.shape[0]:,} rows × {df_tmp.shape[1]} cols")
    else:
        print(f"  pareto_{pareto_idx}: NOT YET COMPLETE")


Summary of completed pickles:
  pareto_682: 3,321 rows × 20 cols
  pareto_701: 3,321 rows × 20 cols
  pareto_708: 3,321 rows × 20 cols
  pareto_709: 3,321 rows × 20 cols
  pareto_710: 3,321 rows × 20 cols
  pareto_717: 3,321 rows × 20 cols
  pareto_718: 3,321 rows × 20 cols
  pareto_719: 3,321 rows × 20 cols
  pareto_727: 3,321 rows × 20 cols
  pareto_728: 3,321 rows × 20 cols
  pareto_898: 3,321 rows × 20 cols
  pareto_917: 3,321 rows × 20 cols
